# S2 — 缺失值不是刪掉就好：Missing Data Strategy

> **時間**：2 小時（講授 70min + 課堂練習 40min + QA 10min）  
> **資料**：`datasets/titanic/train.csv`（講授）、`datasets/house-prices/train.csv`（練習）  
> **學完能幹嘛**：判斷缺值屬於哪種機制，選擇最合適的填補策略，而不是無腦 `dropna()`

---

## 為什麼缺值處理這麼重要？

Kaggle 前 10% 的選手和後 50% 的差距，往往不在模型，而在 **資料前處理**。

新手看到缺值：「刪掉！」→ 丟掉 20% 資料，成績直接掉。  
高手看到缺值：「為什麼缺？缺值本身是不是一種資訊？」

**一句話口訣：缺值有三種死法：隨機缺 (MCAR)、有規律缺 (MAR)、故意缺 (MNAR) — 搞錯了，你的分析就廢了。**

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv('datasets/titanic/train.csv')
print(f'Titanic: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'缺值欄位：')
print(df.isnull().sum()[df.isnull().sum() > 0])

---
## 核心觀念 1／3 — NA 全景圖：看見缺失的模式

在決定怎麼處理之前，先**看清楚**缺值長什麼樣子。三個工具：

| 工具 | 看什麼 |
|:-----|:-------|
| NA% bar chart | 每個欄位缺多少 |
| NA heatmap | 缺值的空間分布 |
| NA correlation | 哪些欄位「一起缺」|

In [ ]:
# 工具 1：NA 比例 bar chart
na_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
na_pct = na_pct[na_pct > 0]

fig, ax = plt.subplots(figsize=(8, 4))
na_pct.plot(kind='bar', color='coral', edgecolor='white', ax=ax)
ax.set_title('缺值比例 (%)', fontsize=14)
ax.set_ylabel('%')
for i, v in enumerate(na_pct):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# 工具 2：NA heatmap（看缺值的「形狀」）
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, ax=ax)
ax.set_title('缺值分布熱力圖（黃色 = 缺值）', fontsize=14)
plt.tight_layout()
plt.show()

print('觀察：')
print('  - Cabin: 大面積黃色 → 系統性缺失')
print('  - Age: 零散分布 → 較隨機')
print('  - Embarked: 幾乎看不到 → 只缺 2 筆')

In [ ]:
# 工具 3：NA 相關性（哪些欄位一起缺？）
# 只看有缺值的欄位
na_cols = df.columns[df.isnull().any()]
na_corr = df[na_cols].isnull().corr()

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(na_corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('缺值相關性矩陣', fontsize=14)
plt.tight_layout()
plt.show()

print('→ Age 和 Cabin 的缺值相關性 ≈ 0.13，幾乎獨立')
print('  如果兩個欄位總是一起缺，通常代表它們有相同的缺失原因')

**口訣**：處理缺值之前，先畫三張圖 — bar chart 看比例、heatmap 看分布、correlation 看關聯。

---
## 核心觀念 2／3 — 三種缺失機制

| 機制 | 全名 | 白話文 | Titanic 範例 |
|:-----|:-----|:-------|:-------------|
| **MCAR** | Missing Completely At Random | 純粹隨機遺失，跟任何欄位都無關 | Embarked（只缺 2 筆） |
| **MAR** | Missing At Random | 缺失跟**其他已知欄位**有關 | Age（3 等艙乘客較容易缺 Age） |
| **MNAR** | Missing Not At Random | 缺失跟**自己的值**有關 | Cabin（沒有艙房的人就沒有 Cabin 號） |

**為什麼要分？** 因為不同機制需要不同策略：
- MCAR → 直接刪或填平均值都安全
- MAR → 需要**分組填補**（groupby + fillna）
- MNAR → 缺值本身就是特徵！要**保留缺失資訊**

In [ ]:
# 範例 1：Embarked — MCAR（純隨機）
print('=== Embarked (MCAR) ===')
print(f'缺值數：{df["Embarked"].isnull().sum()}')
print(f'\n缺值的那 2 筆乘客：')
print(df[df['Embarked'].isnull()][['Name', 'Pclass', 'Fare', 'Survived']])
print('\n→ 這 2 筆乘客看起來沒什麼特別規律，就是登船港口欄位漏填了')
print('→ MCAR 處理方式：用眾數 (mode) 填補即可')

In [ ]:
# 範例 2：Age — MAR（跟其他欄位有關）
print('=== Age (MAR) ===')
print('各 Pclass 的 Age 缺值比例：')
age_na_by_class = df.groupby('Pclass')['Age'].apply(
    lambda x: x.isnull().mean() * 100
)
print(age_na_by_class.round(1))
print('\n→ 3 等艙缺值率 29.3%，1 等艙只有 6.1%')
print('→ Age 的缺失跟 Pclass 有關 — 這就是 MAR')
print('→ MAR 處理方式：按 Pclass 分組，用各組的中位數填補')

In [ ]:
# 範例 3：Cabin — MNAR（缺失本身就是資訊）
print('=== Cabin (MNAR) ===')
df['has_cabin'] = df['Cabin'].notnull().astype(int)

print('有 Cabin vs 沒有 Cabin 的存活率：')
print(df.groupby('has_cabin')['Survived'].mean().round(3))

# 視覺化
fig, ax = plt.subplots(figsize=(6, 4))
df.groupby('has_cabin')['Survived'].mean().plot(
    kind='bar', color=['#ff9999', '#66b2ff'], edgecolor='white', ax=ax
)
ax.set_title('有/沒有 Cabin 的存活率', fontsize=14)
ax.set_xlabel('has_cabin (0=沒有, 1=有)')
ax.set_ylabel('存活率')
ax.set_xticklabels(['No Cabin', 'Has Cabin'], rotation=0)
plt.tight_layout()
plt.show()

print('\n→ 有 Cabin 的人存活率 66.7%，沒有的只有 30.0%')
print('→ Cabin 缺失 ≈ 低等艙 ≈ 低存活率 — 缺失本身就是強特徵！')
print('→ MNAR 處理方式：不要刪掉，建立 has_cabin 二元特徵')

**口訣**：MCAR 隨便填、MAR 分組填、MNAR 當特徵 — 先判斷機制，再選策略。

> ### 💡 知識補給站 — 如何判斷是哪種缺失機制？
> 
> 嚴格來說，三種機制無法「證明」，只能根據領域知識和統計線索**推斷**：
> 
> 1. **分組比較法**：按其他欄位 groupby，看缺值率是否在各組間不同 → 不同就是 MAR
> 2. **缺值 vs target 法**：建立 `is_missing` 二元欄位，看它和 target 的關係 → 有關就可能是 MNAR
> 3. **領域知識**：理解資料的生成過程（例如：沒有泳池的房子不會有泳池面積）
> 
> 實務中，MAR 和 MNAR 的界線模糊。重要的是 **不要用同一種方法處理所有缺值**。
> 
> *延伸關鍵字：Little's MCAR test, missing data mechanism, imputation bias*

---
## 核心觀念 3／3 — 填補策略工具箱

| 策略 | 方法 | 適用場景 |
|:-----|:-----|:--------|
| 刪除列 | `df.dropna()` | 缺值很少（< 5%）且 MCAR |
| 刪除欄 | `df.drop(col)` | 缺值太多（> 80%）且無法利用 |
| 全域填補 | `df[col].fillna(median)` | MCAR，快速但粗糙 |
| 分組填補 | `groupby().transform()` | MAR，按相關欄位分組填 |
| 缺值旗標 | `df[col + '_missing'] = isnull()` | MNAR，保留缺失資訊 |
| 門檻刪除 | `df.dropna(thresh=N)` | 一列中缺太多欄位的刪掉 |

In [ ]:
# 策略實戰：處理 Titanic 三個缺值欄位
df_clean = df.copy()

# 1. Embarked (MCAR) → 用眾數填補
mode_embarked = df_clean['Embarked'].mode()[0]
df_clean['Embarked'] = df_clean['Embarked'].fillna(mode_embarked)
print(f'1. Embarked: 用眾數 "{mode_embarked}" 填補 → 缺值: {df_clean["Embarked"].isnull().sum()}')

In [ ]:
# 2. Age (MAR) → 按 Pclass 分組，用各組中位數填補
print('2. Age: 按 Pclass 分組中位數填補')
print('   各 Pclass 的 Age 中位數：')
age_medians = df_clean.groupby('Pclass')['Age'].median()
print(age_medians)

# groupby + transform 技巧
df_clean['Age'] = df_clean.groupby('Pclass')['Age'].transform(
    lambda x: x.fillna(x.median())
)
print(f'\n   填補後缺值: {df_clean["Age"].isnull().sum()}')

In [ ]:
# 驗證：填補前後 Age 分布有沒有被嚴重扭曲？
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].set_title('填補前 Age 分布', fontsize=12)
sns.kdeplot(df['Age'].dropna(), fill=True, ax=axes[0], color='steelblue')

axes[1].set_title('填補後 Age 分布', fontsize=12)
sns.kdeplot(df_clean['Age'], fill=True, ax=axes[1], color='coral')

plt.tight_layout()
plt.show()
print('→ 分布形狀大致保持，填補策略合理')

In [ ]:
# 3. Cabin (MNAR) → 不填補，改建立 has_cabin 二元特徵
df_clean['has_cabin'] = df_clean['Cabin'].notnull().astype(int)
print(f'3. Cabin: 建立 has_cabin 特徵（不刪除原始 Cabin 欄位，後續可再拆 Deck）')
print(f'   has_cabin 分布: {df_clean["has_cabin"].value_counts().to_dict()}')

In [ ]:
# 最終確認：還有缺值嗎？
remaining_na = df_clean.isnull().sum()
remaining_na = remaining_na[remaining_na > 0]
if len(remaining_na) == 0:
    print('✅ 所有可處理的缺值都已處理完成！')
else:
    print('剩餘缺值：')
    print(remaining_na)

**口訣**：填完一定要驗 — 比較前後分布，確認沒有扭曲原始資料的特性。

> ### 💡 知識補給站 — groupby().transform() vs groupby().apply()
> 
> `transform()` 會**保持原始 index**，輸出和輸入一樣長，適合「填回去」。
> `apply()` 會對每組做聚合，輸出通常比輸入短。
> 
> ```python
> # transform: 891 行 → 891 行（填回原位）
> df.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))
> 
> # apply: 891 行 → 3 行（聚合結果）
> df.groupby('Pclass')['Age'].apply(lambda x: x.median())
> ```
> 
> 分組填補用 `transform`，分組統計用 `apply`。
> 
> *延伸關鍵字：transform vs apply, groupby imputation, index alignment*

---
## 課堂練習（40 min）

用 **House Prices** 資料集練習缺失值策略。

### 🟢 送分題

1. 讀取 `datasets/house-prices/train.csv`
2. 找出所有有缺值的欄位，畫出前 15 個的 NA% bar chart
3. 有幾個欄位缺值超過 50%？

In [ ]:
# TODO: 你的程式碼

### 🟡 核心題

House Prices 中的 `PoolQC`（泳池品質）缺了 99.5%。

1. 這是 MCAR、MAR 還是 MNAR？（提示：看 `PoolArea` 欄位）
2. 檢查：`PoolQC` 缺值的那些房子，`PoolArea` 是多少？
3. 適合的處理策略是什麼？

In [ ]:
# TODO: 你的程式碼

### 🔴 挑戰題

`LotFrontage`（到街道的距離）缺了 259 筆 (17.7%)。

1. 按 `Neighborhood`（社區）分組，看各社區的 `LotFrontage` 缺值率 — 是 MCAR 還是 MAR？
2. 用 `groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))` 做分組填補
3. 畫出填補前後的 KDE 圖，驗證分布是否保持

In [ ]:
# TODO: 你的程式碼

---
## 小結與速查表

| 想做什麼 | 怎麼寫 |
|:---------|:-------|
| 缺值計數 | `df.isnull().sum()` |
| 缺值比例 | `df.isnull().mean() * 100` |
| 缺值熱力圖 | `sns.heatmap(df.isnull(), cbar=False)` |
| 缺值相關性 | `df[na_cols].isnull().corr()` |
| 刪除高缺值列 | `df.dropna(thresh=N)` |
| 全域填補 | `df['col'].fillna(df['col'].median())` |
| 分組填補 | `df.groupby('g')['col'].transform(lambda x: x.fillna(x.median()))` |
| 缺值旗標 | `df['col_missing'] = df['col'].isnull().astype(int)` |
| 眾數填補 | `df['col'].fillna(df['col'].mode()[0])` |

**下節預告 S3**：缺值處理完了，接下來要深入每個欄位的分布 — 哪些是偏態的？哪些有離群值？→ **Distribution & Outlier Analysis**